# 16 — Estudo integrador: MovieLens e feedback sintético de livros

**EP · Data Science Knowledge Base | Sistemas de Recomendação | Notebook 16/16**

**Execução:** Reinicie o kernel e execute as células em ordem. O catálogo de livros e as interações são inteiramente sintéticos.


## Materiais complementares

- [Guia Técnico](reference_materials/16_estudo_integrador_movielens_livros.pdf)
- [Cheat Sheet](cheat_sheets/cheat_sheet_16_estudo_integrador_movielens_livros.pdf)

**Navegação:** os links são relativos à pasta deste notebook.

## Objetivos de aprendizagem
- Comparar um dataset real de filmes com um domínio de livros **inteiramente sintético**
- Separar interação sem nota (feedback implícito) de avaliação explícita
- Calcular um baseline de popularidade suavizada sem confundir as escalas
- Reconhecer os limites de inferência a partir de exemplos artificiais


## Fundamentação conceitual
O **MovieLens** fornece avaliações reais de filmes em escala 0,5–5. O segundo domínio é um exemplo **inventado exclusivamente para ensino**. Nele, definimos `nota = 0` como uma interação sem avaliação explícita, e `nota` de 1 a 10 como avaliação explícita. Essa convenção é uma decisão do exemplo, **não** uma definição universal de feedback implícito. Não interpretar interações sem nota como avaliações negativas ou como valores zero na média de notas.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.metrics.pairwise import cosine_similarity
DATA = Path('datasets')
if not (DATA / 'ratings.csv').exists():
    raise FileNotFoundError('Abra o JupyterLab na pasta da Disciplina 09 (onde existe datasets/).')
ratings = pd.read_csv(DATA / 'ratings.csv')
movies = pd.read_csv(DATA / 'movies.csv')
print('Avaliações:', len(ratings), '| Filmes:', len(movies), '| Usuários:', ratings.userId.nunique())

Avaliações: 100836 | Filmes: 9742 | Usuários: 610


## MovieLens: ranking com suporte

In [2]:
st=ratings.groupby('movieId').rating.agg(['mean','count'])
mu=ratings.rating.mean();st['smooth']=(st['count']*st['mean']+30*mu)/(st['count']+30)
print(movies.set_index('movieId').loc[st.sort_values('smooth',ascending=False).head(5).index,'title'].tolist())

['Shawshank Redemption, The (1994)', 'Godfather, The (1972)', 'Fight Club (1999)', 'Star Wars: Episode IV - A New Hope (1977)', 'Usual Suspects, The (1995)']


**Interpretação.** O baseline de popularidade suavizada pode ser transposto para exemplos em outros domínios, desde que as unidades, o significado dos eventos e os parâmetros sejam definidos separadamente.


## Catálogo e interações de livros — dados inteiramente sintéticos


In [3]:
# Dados criados pelo autor exclusivamente para fins didáticos.
# Não há leitura de arquivos externos de livros ou dados de pessoas reais.
origem_livros = "sintetico_autoral"
catalogo_livros = pd.DataFrame({
    "livro_id": ["SYN-001", "SYN-002", "SYN-003", "SYN-004", "SYN-005"],
    "titulo": ["Livro A (fictício)", "Livro B (fictício)", "Livro C (fictício)",
               "Livro D (fictício)", "Livro E (fictício)"],
})
interacoes_livros = pd.DataFrame({
    "usuario_id": [1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 6, 6],
    "livro_id": ["SYN-001", "SYN-002", "SYN-001", "SYN-003",
                 "SYN-002", "SYN-004", "SYN-001", "SYN-005",
                 "SYN-003", "SYN-004", "SYN-002", "SYN-005"],
    "nota": [8, 0, 9, 7, 6, 0, 10, 8, 0, 5, 7, 9],
})
assert interacoes_livros["livro_id"].isin(catalogo_livros["livro_id"]).all()
assert interacoes_livros["nota"].between(0, 10).all()
assert interacoes_livros["nota"].gt(0).any()
print("Fonte: DADOS SINTÉTICOS AUTORAIS | Interações:", len(interacoes_livros),
      "| Livros:", len(catalogo_livros))
display(interacoes_livros.head())


Fonte: DADOS SINTÉTICOS AUTORAIS | Interações: 12 | Livros: 5


,usuario_id,livro_id,nota
0,1,SYN-001,8
1,1,SYN-002,0
2,2,SYN-001,9
3,2,SYN-003,7
4,3,SYN-002,6


**Interpretação.** Todos os registros são criados no código e não derivam de uma base externa. Um evento com `nota = 0` significa somente que houve interação sem nota; seu significado foi definido para esta demonstração.


## Separar interações sem nota e avaliações explícitas


In [4]:
interacoes_sem_nota = interacoes_livros.loc[interacoes_livros["nota"].eq(0)].copy()
avaliacoes_explicitas = interacoes_livros.loc[interacoes_livros["nota"].gt(0)].copy()
print("Interações sem nota (código 0):", len(interacoes_sem_nota),
      "| Avaliações explícitas (1–10):", len(avaliacoes_explicitas))
assert avaliacoes_explicitas["nota"].between(1, 10).all()
assert len(interacoes_sem_nota) + len(avaliacoes_explicitas) == len(interacoes_livros)


Interações sem nota (código 0): 3 | Avaliações explícitas (1–10): 9


**Interpretação.** Não inclua o código `0` no cálculo da média de avaliações explícitas. Um evento sem nota é informação comportamental, não uma nota baixa.


## Ranking sintético com avaliações explícitas e suavização


In [5]:
resumo_notas = avaliacoes_explicitas.groupby("livro_id")["nota"].agg(["mean", "count"])
media_geral_livros = avaliacoes_explicitas["nota"].mean()
resumo_notas["score"] = (resumo_notas["mean"] * resumo_notas["count"]
                         + 10 * media_geral_livros) / (resumo_notas["count"] + 10)
ranking_livros = (resumo_notas.sort_values("score", ascending=False)
                  .head(8).reset_index()
                  .merge(catalogo_livros, on="livro_id", how="left"))
assert ranking_livros["titulo"].notna().all()
print(ranking_livros[["titulo", "count", "score"]].to_string(index=False))


            titulo  count    score
Livro A (fictício)      3 7.974359
Livro E (fictício)      2 7.805556
Livro C (fictício)      1 7.606061
Livro B (fictício)      2 7.472222
Livro D (fictício)      1 7.424242


**Interpretação.** O ranking demonstra a combinação de média e tamanho amostral. Como todas as interações são fictícias e pequenas, não se pode concluir nada sobre preferências de leitores reais, qualidade preditiva ou desempenho em produção.


## Aprofundamento e aplicação

**Ponto de atenção deste capítulo:** separação de domínios. Este complemento não substitui as seções e exemplos anteriores.

**Verificação conceitual.** Explique, com suas palavras, em qual etapa do fluxo de trabalho o tema **separação de domínios** pode produzir conclusões enganosas. Indique um indicador observável, um teste mínimo e uma limitação do procedimento.

**Exercício de extensão.** Reproduza o experimento central deste notebook alterando **um parâmetro por vez**. Compare com uma linha de base simples, reporte a variação da métrica pertinente e justifique o resultado. Quando houver treino/teste, qualquer etapa que aprenda parâmetros deve usar apenas o treino.

**Checklist de conclusão:** (1) entradas identificadas; (2) sementes registradas; (3) saídas interpretadas; (4) premissas explicitadas; (5) dados originais não sobrescritos.


### Laboratório adicional — código zero versus nota explícita (exemplo sintético)

Experimento independente do estado das células anteriores. Explique por que a convenção adotada para um evento sem nota não deve ser generalizada a outros datasets.


In [6]:
import pandas as pd
exemplo = pd.DataFrame({"livro": ["A", "B", "C"], "nota": [0, 7, 10]})
notas_explicitas = exemplo.loc[exemplo["nota"].gt(0)]
print("Interações totais:", len(exemplo), "| Notas explícitas:", len(notas_explicitas))
assert len(notas_explicitas) == 2


Interações totais: 3 | Notas explícitas: 2


### Exercício resolvido — protocolo mínimo de validação

**Enunciado.** Proponha uma comparação para **separação de domínios** sem contaminar a avaliação.

**Solução orientadora.** Defina uma hipótese falsificável, escolha um baseline simples, fixe uma divisão adequada ao problema, altere uma única decisão metodológica e compare métricas no conjunto reservado. Não transforme exemplos sintéticos em conclusões sobre populações reais. Documente a configuração e os casos de falha. Quando o tema não envolver predição, substitua o baseline por uma implementação de referência e compare correção, cobertura e custo.


## Análise crítica — Separação de domínios e tipos de feedback

O MovieLens representa avaliações explícitas na escala 0,5–5. O exemplo fictício de livros usa avaliações 1–10 e define zero como evento sem nota. Não há equivalência automática entre as escalas nem entre dados reais e sintéticos. A demonstração apenas mostra como distinguir tipos de evento e calcular um baseline de maneira coerente; não permite estimar desempenho em usuários reais.

**Origem:** exemplo didático autoral, sem transcrição de dados de terceiros.


In [7]:
qtd_sem_nota = int(interacoes_livros["nota"].eq(0).sum())
qtd_com_nota = int(interacoes_livros["nota"].between(1, 10).sum())
print({"interacoes_sem_nota": qtd_sem_nota,
       "avaliacoes_explicitas_1_a_10": qtd_com_nota,
       "total": len(interacoes_livros), "origem": origem_livros})
assert qtd_sem_nota + qtd_com_nota == len(interacoes_livros)


{'interacoes_sem_nota': 3, 'avaliacoes_explicitas_1_a_10': 9, 'total': 12, 'origem': 'sintetico_autoral'}


### Critério de leitura

Interprete os resultados como diagnóstico didático. Para comparar modelos de recomendação, defina previamente população, itens elegíveis, janela temporal, sinal de relevância, baseline e métrica; mantenha treino, validação e teste separados.

**Atividade:** registre o efeito de uma mudança de parâmetro e explique uma ameaça à validade do experimento.

## Limitações e boas práticas
O experimento sobre livros é integralmente sintético. Não representa qualquer amostra populacional e não valida um recomendador em produção. Nunca compare diretamente as notas entre escalas ou misture eventos sem nota a valores de avaliação.


## Exercícios
Explique por que a nota 4 no MovieLens não deve ser comparada diretamente à nota 4 no exemplo sintético de livros. Por que o código zero não é uma avaliação de valor zero?


## Soluções comentadas
As escalas diferem (MovieLens 0,5–5; exemplo de livros 1–10), e as populações não são comparáveis (MovieLens real; livros fictícios). O código zero indica uma interação sem avaliação explícita **por convenção deste exemplo**, portanto não pertence à média de notas.


## Síntese
Reveja os resultados e compare métodos usando um protocolo de dados e métricas consistente. Não trate avaliações ausentes como avaliações negativas.

### Exportação de resultados

A próxima célula grava dois arquivos CSV didáticos em `outputs/`, ambos com sufixo `_sintetico` e campo `origem`, para impedir interpretação como observações reais. Não são publicados automaticamente no GitHub.


In [8]:
from pathlib import Path
out = Path("outputs")
out.mkdir(parents=True, exist_ok=True)
ranking_publicavel = (ranking_livros[["livro_id", "titulo", "count", "score"]]
                      .rename(columns={"count": "n_notas_explicitas",
                                       "score": "media_suavizada_explicita"})
                      .assign(origem=origem_livros))
ranking_publicavel.to_csv(out / "16_ranking_livros_sintetico.csv", index=False)
resumo_feedback = pd.DataFrame([{
    "interacoes_sem_nota": len(interacoes_sem_nota),
    "avaliacoes_explicitas": len(avaliacoes_explicitas),
    "total_interacoes": len(interacoes_livros),
    "origem": origem_livros,
}])
resumo_feedback.to_csv(out / "16_resumo_feedback_livros_sintetico.csv", index=False)
print("Outputs sintéticos:", sorted(p.name for p in out.glob("16_*.csv")))


Outputs sintéticos: ['16_ranking_livros_sintetico.csv', '16_resumo_feedback_livros_sintetico.csv']
